<a href="https://www.kaggle.com/code/alexvmt/preprocess-images-for-terainet?scriptVersionId=246822872" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Preprocess images for TeraiNet

1. Run MegaDetector on all images
2. Snip images
3. Copy snipped images to Kaggle Output

*Note: Images must have been previously downloaded to Drive via Colab and then uploaded to Kaggle as a dataset.*

## Setup

### Change working directory

In [ ]:
%cd ../../

### Clone TeraiNet repo

In [ ]:
!git clone https://github.com/alexvmt/terainet.git

### MegdaDetector installs

In [ ]:
!pip install megadetector

In [ ]:
!wget -O visualization_utils.py "https://raw.githubusercontent.com/agentmorris/MegaDetector/refs/heads/main/megadetector/visualization/visualization_utils.py"

### Imports

In [ ]:
project_dir = "terainet"
print("Installing TeraiNet package...")
!pip install -e "$project_dir"

In [ ]:
import os

from terainet import load_config

### Set variables and parameters

In [ ]:
config_path = os.path.join(project_dir, "config.yaml")
config = load_config(config_path)

# scripts
scripts_dir = os.path.join(project_dir, config["sampling_downloading"]["scripts_dir"])
run_md_script = os.path.join(scripts_dir, config["scripts"]["run_megadetector_script"])
copy_snipped_images_script = os.path.join(
    scripts_dir, config["scripts"]["copy_snipped_images_script"]
)

# md
md_dir = config["preprocessing"]["megadetector_dir"]
!mkdir -p "$md_dir"
md_file = config["preprocessing"]["megadetector_file"]
md_out_file = config["preprocessing"]["megadetector_out_file"]

# images dir
images_input_dir = config["preprocessing"]["images_input_dir"]
images_output_dir = config["preprocessing"]["images_output_dir"]
!mkdir -p "$images_output_dir"

# num classes
num_classes = config["classes"]["num_classes"]

# set parameters for snipping images
INPUT_DIR = images_input_dir
MD_FILE = md_out_file
SNIP_DIR = config["preprocessing"]["snip_dir"]
LOWER_CONF = 0.6
SNIP_SIZE = config["preprocessing"]["snip_size"]

## Run MegaDetector

In [ ]:
# download megadetector model file
!wget -O "$md_dir/$md_file" https://github.com/agentmorris/MegaDetector/releases/download/v5.0/md_v5a.0.0.pt

In [ ]:
# run megadetector
!time python "$run_md_script" "$images_input_dir" "$md_dir/$md_file" "$md_dir"

## Snip images
Follow [mewc-snip](https://github.com/zaandahl/mewc-snip)

## Copy snipped images to Kaggle Ouput

In [ ]:
# create target directories
!mkdir -p "$images_output_dir/train"
!mkdir -p "$images_output_dir/val"
!mkdir -p "$images_output_dir/test"
!mkdir -p "$images_output_dir/test2"

In [ ]:
# copy snipped images to target directories
!time bash "$copy_snipped_images_script" "snips" "$images_output_dir" "$num_classes"

In [ ]:
# check file count per class in target directories
directories_to_check = ["train", "val", "test", "test2"]

for dir_name in directories_to_check:
    dir_path = os.path.join(images_output_dir, dir_name)

    if os.path.exists(dir_path):
        print(f"Counting files in subdirectories of {dir_name}:")
        subdirs = sorted(os.listdir(dir_path), key=lambda x: int(x.split("_")[1]))

        for subdir in subdirs:
            subdir_path = os.path.join(dir_path, subdir)

            if os.path.isdir(subdir_path):
                file_count = len(
                    [
                        f
                        for f in os.listdir(subdir_path)
                        if os.path.isfile(os.path.join(subdir_path, f))
                    ]
                )
                print(f"{subdir}: {file_count}")

    else:
        print(f"Directory not found: {dir_path}")